# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading, exploring, and processing the FAIR$^2$ dataset using the [mlcroissant](https://mlcommons.github.io/croissant/) library, referencing all data entities by their `@id`.

### Dataset Source
The dataset is defined by a Croissant schema and can be accessed using the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare to explore the available record sets and fields. All entity references in subsequent steps will use their Croissant `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
print("Loaded dataset:")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")


## 2. Data Overview
Review the available record sets, their `@id`s, and the associated fields in the dataset. All entities are referenced by their `@id`, following the Croissant specification.

In [ ]:
# List all record sets in the dataset with their @id
record_set_objs = list(dataset.metadata.record_sets)

print("Available record sets:")
for rs in record_set_objs:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    name: {rs.name}")
    print(f"    description: {getattr(rs, 'description', '')}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', '')}")
    print()
# Save first RecordSet id for later use
if record_set_objs:
    first_record_set_id = record_set_objs[0].id
else:
    first_record_set_id = None

## 3. Data Extraction
We'll read the records from a RecordSet using its `@id`. Use this step to extract data from each RecordSet into a DataFrame for further analysis.

In [ ]:
# Collect all RecordSet @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

print(f"Record sets to extract: {record_set_ids}")
dataframes = {}
# Loop through all record sets to extract records into DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    if not df.empty:
        dataframes[record_set_id] = df
        print(f"\nDataFrame for RecordSet @id '{record_set_id}':")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"RecordSet @id '{record_set_id}' yielded no records.")
# For reference, pick one record set with data
main_record_set_id = next(iter(dataframes)) if dataframes else None

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field (referenced by its `@id`) and demonstrate common data processing steps such as filtering, normalization, and grouping. All entity references use their respective `@id`s.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        # Pick first numeric field for demonstration
        numeric_field_id = numeric_field_candidates[0]  # this is the @id
        print(f"Analyzing numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in ['float64', 'int64'] else 0  # Example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field (if any exists)
        categorical_field_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
        if categorical_field_candidates:
            group_field_id = categorical_field_candidates[0]  # Use @id
            print(f"Grouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields found in the main RecordSet data.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's plot the distribution of a numeric field in one of the RecordSets using its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_candidates:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No data or numeric column available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load and explore the FAIR$^2$ dataset defined by its Croissant schema. We referenced all record sets and fields by their official `@id`, loaded multiple record sets into DataFrames, performed basic EDA including filtering and normalization on numeric fields, and visualized data distributions.

This approach ensures reproducibility and clarity when working with richly described datasets. For more advanced analyses, consider exploring additional RecordSets or combining fields across entities using their persistent `@id` references.